## Read Data from Bronze Layer

In [0]:
%run ./test

In [0]:
def load_data(table):
  return spark.read.table(table)


## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_load_data'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Get Star Players from Common Player Info

By filtering the greatest 75th anniversay team players

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col
import pyspark.sql.functions as F

def get_star_players(common_player_info_df: DataFrame):
    return common_player_info_df.where(col("greatest_75_flag") == "Y").withColumn("player_id", col("person_id")).drop("person_id").select("player_id", "first_name", "last_name", "display_first_last", "position", "from_year", "to_year")

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_get_star_players'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Game with Star Absence

Get the games that the star players are off

In [0]:
def process_game_with_star_absence(star_players_df: DataFrame, inactive_players_df: DataFrame):
    return star_players_df.join(inactive_players_df, on="player_id")
    # .select("game_id", "player_id", "team_id", "team_name")


## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_process_game_with_star_absence'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Game without Star Players

Get all games with star players off the court.

Only consider the regular season games

In [0]:
def process_game_with_winning_and_losing(game_with_star_absence_df: DataFrame, game_df: DataFrame):
    return game_with_star_absence_df.join(game_df, on="game_id").where(col("season_type") == "Regular Season")

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_procee_game_with_winning_and_losing'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Games with Star Players

Get all games with star players on the court

### Get team id for star players

In [0]:
def get_team_id_for_star_players(inactive_players: DataFrame, star_players: DataFrame):
    return star_players.join(inactive_players, on="player_id")\
        .dropDuplicates(["team_id", "player_id"])

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_get_team_id_for_star_players'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### Filter Games within the Star Player Life Span

These games include those where the star players were on the court and those where they were not.

In [0]:


def get_games_from_team_id_within_star_player_life_span(team_player_df: DataFrame, game_df: DataFrame):
    team_player_game_home_df = team_player_df\
        .join(game_df, on=(team_player_df["team_id"] == col("team_id_home")) | (team_player_df["team_id"] == col("team_id_away")))

    team_player_game_df = team_player_game_home_df\
        .where(col("season_type") == "Regular Season")\
            .where(F.year(F.to_date(col("game_date"))) >= F.col("from_year").cast("int"))\
                .where(F.year(F.to_date(col("game_date"))) <= F.col("to_year").cast("int"))

    return team_player_game_df

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_get_games_from_team_id_within_star_player_life_span'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

### Filter Games with the Star Players were on the Court

In [0]:
def get_games_with_star_players_on_court(star_player_all_game_df: DataFrame, inactive_players_df: DataFrame):
    return star_player_all_game_df.join(
        inactive_players_df,
        on=["game_id", "player_id"],
        how="left_anti"
        )

## Unit Test

In [0]:
suite = unittest.TestSuite()

suite.addTest(TestPlayerPipeline('test_get_games_with_star_players_on_court'))

runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## Process Game 

In [0]:
INACTIVE_PLAYERS_TABLE = "nba_insights.bronze.inactive_players"
COMMON_PLAYER_INFO_TABLE = "nba_insights.bronze.common_player_info"
GAME_TABLE = "nba_insights.bronze.games"

inactive_players_df = load_data(INACTIVE_PLAYERS_TABLE)
common_player_info_df = load_data(COMMON_PLAYER_INFO_TABLE)
game_df = load_data(GAME_TABLE)

def process_game_with_star_off_court(common_player_info_df: DataFrame, game_df: DataFrame, inactive_players_df: DataFrame):
    star_df = common_player_info_df.transform(get_star_players)
    game_with_star_absence_df = star_df.transform(process_game_with_star_absence, inactive_players_df)
    game_stats_with_star_absence_df = game_with_star_absence_df.transform(process_game_with_winning_and_losing, game_df)\
        .withColumn("star_team_id", col("team_id"))\
            .drop("team_id")\
                .select("game_id", "star_team_id", "team_id_home", "wl_home", "team_id_away", "wl_away", "player_id")

    return game_stats_with_star_absence_df

def process_game_with_star_on_court(common_player_info_df: DataFrame, game_df: DataFrame, inactive_players_df: DataFrame):
    star_df = common_player_info_df.transform(get_star_players)
    team_id_for_star_players = inactive_players_df.transform(get_team_id_for_star_players, star_df)
    display(team_id_for_star_players)
    star_player_all_games_df = team_id_for_star_players.transform(get_games_from_team_id_within_star_player_life_span, game_df)
    display(star_player_all_games_df)
    games_with_star_on_court_df = star_player_all_games_df.transform(get_games_with_star_players_on_court, inactive_players_df)

    return games_with_star_on_court_df


INACTIVE_PLAYERS_TABLE = "nba_insights.bronze.inactive_players"
COMMON_PLAYER_INFO_TABLE = "nba_insights.bronze.common_player_info"
GAME_TABLE = "nba_insights.bronze.game"

# game_stats_with_star_absence_df = process_game_with_star_off_court(common_player_info_df, game_df, inactive_players_df)
# display(game_stats_with_star_absence_df)

games_with_star_on_court_df = process_game_with_star_on_court(common_player_info_df, game_df, inactive_players_df)
display(games_with_star_on_court_df)


## E2E Test

In [0]:
# E2E Test

## Persist Silver Layer

In [0]:
common_player_info_df = load_data(COMMON_PLAYER_INFO_TABLE)
star_players_df = get_star_players(common_player_info_df)

star_players_df = common_player_info_df.transform(get_star_players)
star_players_df.write.mode("overwrite").saveAsTable("nba_insights.silver.star_players")

game_stats_with_star_absence_df.write.mode("overwrite").saveAsTable("nba_insights.silver.games_stats_with_stars_absence")